In [2]:
import pandas as pd
import joblib
import nltk
import string
import time
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from Preprocessing_pipeline import normalize_english

# 1. Download required NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)

# 2. Setup preprocessing parameters
english_StopWords = set(stopwords.words('english'))
punctuations = set(string.punctuation)
lemmatizer = WordNetLemmatizer()

# 3. Load & prepare dataset
df = pd.read_csv('../data/imdb_reviews.csv')
#df = df.head(1000).copy()
df = df.dropna(subset=['review', 'sentiment']).copy()

df['clean_text'] = df['review'].apply(
    normalize_english,
    lemmatizer=lemmatizer,
    punctuations=punctuations,
    english_StopWords=english_StopWords
)

df['sentiment'] = df['sentiment'].astype(str).str.strip().str.capitalize()

# 4. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], 
    df['sentiment'], 
    test_size=0.2, 
    random_state=42
)

# 5. Define Classifiers to Compare
classifiers = {
    'Logistic Regression': LogisticRegression(C=2.0, max_iter=1000),
    'Multinomial Naive Bayes': MultinomialNB(alpha=1.0),
    'Linear SVM': LinearSVC(C=1.0)
}

best_model = None
best_accuracy = 0.0
best_model_name = ""

# 6. Evaluate Classifiers
print("\n=== English Model Comparison Results ===")
for name, clf in classifiers.items():
    model = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=20000)),
        ('clf', clf)
    ])
    
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"\n--- {name} ---")
    print(f"Training Time: {train_time:.4f} sec")
    print(f"Accuracy: {acc * 100:.2f}%")
    print(classification_report(y_test, y_pred, zero_division=0))
    
    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model
        best_model_name = name

# 7. Save the Best Model
print(f"\nBest Classifier Selected: {best_model_name} with Accuracy: {best_accuracy * 100:.2f}%")
joblib.dump(best_model, 'English_model_weights.pkl')
print("Best model saved to English_model_weights.pkl")


=== English Model Comparison Results ===

--- Logistic Regression ---
Training Time: 7.2831 sec
Accuracy: 89.16%
              precision    recall  f1-score   support

    Negative       0.90      0.88      0.89      2481
    Positive       0.88      0.91      0.89      2519

    accuracy                           0.89      5000
   macro avg       0.89      0.89      0.89      5000
weighted avg       0.89      0.89      0.89      5000


--- Multinomial Naive Bayes ---
Training Time: 6.8247 sec
Accuracy: 87.08%
              precision    recall  f1-score   support

    Negative       0.88      0.86      0.87      2481
    Positive       0.86      0.88      0.87      2519

    accuracy                           0.87      5000
   macro avg       0.87      0.87      0.87      5000
weighted avg       0.87      0.87      0.87      5000


--- Linear SVM ---
Training Time: 7.4634 sec
Accuracy: 88.78%
              precision    recall  f1-score   support

    Negative       0.90      0.88     